In [0]:
for q in spark.streams.active:
    print("Stopping:", q.name, q.id)
    q.stop()

print("All active streaming queries stopped.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    DoubleType, LongType
)

# ============================================================
# REAL-TIME STOCK PIPELINE
# NiFi → S3 Bronze → Databricks → Silver → Gold
# Compatible with current Databricks compute
# ============================================================

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BRONZE_PATH = "s3://stocks-bronze-layer/raw/realtime_finnhub/"

SILVER_PATH = "s3://stocks-silver-layer/processed/realtime/"

CHECKPOINT_PATH = (
    "s3://stocks-bronze-layer/"
    "checkpoints/realtime_stock_pipeline/"
)

GOLD_FACT_PATH = (
    "s3://stocks-gold-layer/"
    "fact_stock_prices_realtime/"
)

GOLD_DATE_PATH = (
    "s3://stocks-gold-layer/"
    "dim_date_realtime/"
)

# ------------------------------------------------------------
# 2. FINNHUB RAW SCHEMA
# ------------------------------------------------------------

raw_schema = StructType([
    StructField("c", DoubleType(), True),
    StructField("d", DoubleType(), True),
    StructField("dp", DoubleType(), True),
    StructField("h", DoubleType(), True),
    StructField("l", DoubleType(), True),
    StructField("o", DoubleType(), True),
    StructField("pc", DoubleType(), True),
    StructField("t", LongType(), True)
])

# ------------------------------------------------------------
# 3. AUTO LOADER
# Reads ONLY NEW files from NiFi Bronze
# ------------------------------------------------------------

bronze_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            CHECKPOINT_PATH + "schema/"
        )
        .schema(raw_schema)
        .load(BRONZE_PATH)
)

# ------------------------------------------------------------
# 4. REAL-TIME CLEANING / PREPROCESSING
# ------------------------------------------------------------

silver_stream = (
    bronze_stream

    # Unix timestamp → timestamp
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.from_unixtime(F.col("t"))
        )
    )

    # Date
    .withColumn(
        "event_date",
        F.to_date(F.col("event_timestamp"))
    )

    # YYYYMMDD key
    .withColumn(
        "date_key",
        F.date_format(
            F.col("event_timestamp"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Rename Finnhub fields
    .withColumnRenamed("c", "close")
    .withColumnRenamed("d", "daily_change")
    .withColumnRenamed("dp", "daily_return")
    .withColumnRenamed("h", "high")
    .withColumnRenamed("l", "low")
    .withColumnRenamed("o", "open")
    .withColumnRenamed("pc", "previous_close")

    # Validation
    .filter(
        F.col("close").isNotNull()
        & F.col("event_timestamp").isNotNull()
        & F.col("high").isNotNull()
        & F.col("low").isNotNull()
    )

    # Derived metric
    .withColumn(
        "daily_range",
        F.col("high") - F.col("low")
    )
)

# ------------------------------------------------------------
# 5. WRITE SILVER
# AvailableNow = process currently available files and stop
# ------------------------------------------------------------

silver_query = (
    silver_stream
        .writeStream
        .format("parquet")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH + "silver/"
        )
        .trigger(availableNow=True)
        .start(SILVER_PATH)
)

# ------------------------------------------------------------
# 6. REAL-TIME GOLD FACT
# ------------------------------------------------------------

gold_fact_stream = (
    silver_stream.select(
        "date_key",
        "event_timestamp",
        "event_date",
        "open",
        "high",
        "low",
        "close",
        "previous_close",
        "daily_change",
        "daily_return",
        "daily_range"
    )
)

gold_fact_query = (
    gold_fact_stream
        .writeStream
        .format("parquet")
        .outputMode("append")
        .partitionBy("date_key")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH + "gold_fact/"
        )
        .trigger(availableNow=True)
        .start(GOLD_FACT_PATH)
)

# ------------------------------------------------------------
# 7. REAL-TIME DATE DIMENSION
# ------------------------------------------------------------

date_stream = (
    silver_stream
        .select(
            "date_key",
            "event_date"
        )
        .dropDuplicates(["date_key"])
)

gold_date_query = (
    date_stream
        .writeStream
        .format("parquet")
        .outputMode("append")
        .partitionBy("date_key")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH + "gold_date/"
        )
        .trigger(availableNow=True)
        .start(GOLD_DATE_PATH)
)

# ------------------------------------------------------------
# 8. WAIT FOR ALL THREE QUERIES TO FINISH
# ------------------------------------------------------------

print("====================================================")
print("REAL-TIME PIPELINE STARTED")
print("====================================================")
print("Bronze :", BRONZE_PATH)
print("Silver :", SILVER_PATH)
print("Gold Fact :", GOLD_FACT_PATH)
print("Gold Date :", GOLD_DATE_PATH)
print("Trigger : AvailableNow")
print("====================================================")

silver_query.awaitTermination()
gold_fact_query.awaitTermination()
gold_date_query.awaitTermination()

print("====================================================")
print("REAL-TIME PIPELINE COMPLETED")
print("All currently available files were processed.")
print("====================================================")